# LeetCode 3939 — Count Non Adjacent Subsets in a Rooted Tree bằng Python

- Link: https://leetcode.com/problems/count-non-adjacent-subsets-in-a-rooted-tree/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên cây (tree DP), tập độc lập, knapsack theo số dư, tích chập vòng (cyclic convolution), modulo `10^9 + 7`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countValidSubsets(self, parent: List[int], nums: List[int], k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho cây có gốc `0` (mỗi nút `i ≥ 1` có cha `parent[i] < i`), giá trị `nums[i]` và số `k`. Một tập con **khác rỗng** các nút gọi là **hợp lệ** nếu:

- tổng giá trị các nút được chọn **chia hết cho `k`**, và
- **không có hai nút kề nhau** (cha – con) cùng được chọn.

Đếm số tập hợp lệ, lấy modulo `10^9 + 7`.

Đây là bài "đếm tập độc lập trên cây" kết hợp "knapsack theo số dư". Mỗi cây con sẽ lưu **số cách chọn theo từng số dư** của tổng, và hai cây con anh em được ghép bằng **tích chập vòng**.

Sau notebook này bạn sẽ:

1. Hiểu trạng thái `f0[v][s]` / `f1[v][s]` = số cách chọn trong cây con của `v` với tổng `≡ s (mod k)` khi `v` **không chọn / có chọn**.
2. Biết ghép các con bằng tích chập vòng (cộng chỉ số **modulo `k`**).
3. Tận dụng điều kiện `parent[i] < i` để duyệt từ `n−1` về `0` **không cần đệ quy**.
4. Nhớ **trừ tập rỗng** ở cuối (tập rỗng có tổng 0, chia hết cho mọi `k`).
5. Viết 3 lời giải (vét cạn tập con, DP đệ quy với tích chập ngây thơ, DP lặp với tích chập bằng phép dịch vòng) và đối chiếu bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cây gốc `0`, `parent[0] = −1`, `parent[i] < i` với `i ≥ 1`. Mảng `nums`, số `k`. Một tập con khác rỗng là hợp lệ nếu tổng chia hết cho `k` và không chứa cặp nút (nút, cha trực tiếp của nó). Đếm số tập hợp lệ modulo `10^9 + 7`.

**Ví dụ 1:** `parent = [-1,0,1]`, `nums = [1,2,3]`, `k = 3` → `1` (chỉ `{2}`).

**Ví dụ 2:** `parent = [-1,0,0,0]`, `nums = [2,1,2,1]`, `k = 3` → `2` (`{1,2}` và `{2,3}`).

**Ràng buộc:** `1 ≤ n ≤ 1000`, `1 ≤ nums[i] ≤ 10^9`, `1 ≤ k ≤ 100`.

In [1]:
import random, time, sys
from itertools import combinations

MOD = 10**9 + 7
EXAMPLES = [
    (([-1, 0, 1], [1, 2, 3], 3), 1),
    (([-1, 0, 0, 0], [2, 1, 2, 1], 3), 2),
]
for args, exp in EXAMPLES:
    print(args, "-> kỳ vọng", exp)

([-1, 0, 1], [1, 2, 3], 3) -> kỳ vọng 1
([-1, 0, 0, 0], [2, 1, 2, 1], 3) -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Tập độc lập trên cây

"Không có hai nút kề nhau" gọi là **tập độc lập** (independent set). Với cây, quyết định cho nút `v` chỉ ràng buộc với **cha** và **các con** của nó. Nếu `v` được chọn thì mọi con **không** được chọn; nếu `v` không chọn thì mỗi con **tự do**.

In [2]:
# Cây 0 -> 1 -> 2 (đường thẳng): liệt kê mọi tập độc lập
parent = [-1, 0, 1]
n = 3
for mask in range(1 << n):
    S = [i for i in range(n) if mask >> i & 1]
    ok = all(parent[v] not in S for v in S)
    if ok:
        print(S)

[]
[0]
[1]
[2]
[0, 2]


### 3.2. Tách theo số dư mod `k`

Ta chỉ cần biết tổng **modulo `k`**. Với mỗi cây con lưu mảng `cnt[s]` (độ dài `k`) = số cách có tổng `≡ s (mod k)`. Chọn thêm nút `v` giá trị `x` biến chỉ số `s` thành `(s + x) % k` — một **dịch vòng** của mảng.

In [3]:
k = 5
cnt = [1, 0, 2, 0, 0]               # 1 cách có tổng ≡ 0, 2 cách có tổng ≡ 2
x = 7                                # chọn thêm nút giá trị 7 (7 % 5 = 2)
shifted = [0] * k
for s in range(k):
    shifted[(s + x) % k] += cnt[s]
print(shifted)                       # [0, 0, 1, 0, 2]
r = x % k
print(cnt[-r:] + cnt[:-r])           # cách viết gọn bằng cắt lát: cùng kết quả

[0, 0, 1, 0, 2]
[0, 0, 1, 0, 2]


### 3.3. Tích chập vòng (cyclic convolution)

Ghép hai nhóm độc lập: nhóm `A` có `a[i]` cách cho số dư `i`, nhóm `B` có `b[j]` cách cho số dư `j`. Số cách ghép có tổng số dư `t` là `Σ_{i + j ≡ t} a[i]·b[j]` — gọi là tích chập vòng độ dài `k`. Cài đặt đơn giản: hai vòng lặp, `c[(i + j) % k] += a[i]*b[j]` — `O(k²)`.

In [4]:
def cyc_conv(a, b, k):
    c = [0] * k
    for i in range(k):
        if a[i]:
            for j in range(k):
                c[(i + j) % k] = (c[(i + j) % k] + a[i] * b[j]) % MOD
    return c

# "1 cách tổng 0 hoặc 1 cách tổng 1" ghép với "1 cách tổng 0 hoặc 1 cách tổng 2" (k = 3)
print(cyc_conv([1, 1, 0], [1, 0, 1], 3))   # tổng 0:1, 1:1, 2:1, 3≡0:1  -> [2, 1, 1]

[2, 1, 1]


### 3.4. Thứ tự xử lý khi `parent[i] < i`

Vì cha luôn có chỉ số nhỏ hơn con, duyệt `i` từ `n−1` về `0` bảo đảm **mọi con của `i` đã được xử lý trước `i`**. Không cần dựng danh sách con hay đệ quy — tránh giới hạn độ sâu đệ quy khi cây là đường thẳng `n = 1000`.

In [5]:
parent = [-1, 0, 0, 1, 1, 2]
for i in range(len(parent) - 1, 0, -1):
    print(f"xử lý nút {i}, rồi gộp vào cha {parent[i]}")

xử lý nút 5, rồi gộp vào cha 2
xử lý nút 4, rồi gộp vào cha 1
xử lý nút 3, rồi gộp vào cha 1
xử lý nút 2, rồi gộp vào cha 0
xử lý nút 1, rồi gộp vào cha 0


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2**: gốc `0` giá trị `2`, ba con `1, 2, 3` giá trị `1, 2, 1`, `k = 3`.

Không chọn gốc: mỗi lá chọn tùy ý (chọn hoặc không). Chọn gốc: không chọn lá nào, tổng `2` — không chia hết 3.

Các tập con của ba lá và tổng:

| Tập | Tổng | Chia hết 3? |
|---|---|---|
| ∅ | 0 | có, nhưng rỗng → **loại** |
| `{1}` | 1 | không |
| `{2}` | 2 | không |
| `{3}` | 1 | không |
| `{1,2}` | 3 | **có** |
| `{1,3}` | 2 | không |
| `{2,3}` | 3 | **có** |
| `{1,2,3}` | 4 | không |

Đáp án `2` ✓. Nhận xét: đếm theo từng tổng cho hai/ba lá là một bài **ghép** (tích chập) của các hàm "số cách theo số dư" của từng lá: `[1, 0, 0] + [0, 1, 0]` (lá giá trị 1: không chọn → số dư 0; chọn → số dư 1)...

In [6]:
# Mỗi lá giá trị x có hàm đếm [không chọn, chọn] theo số dư mod 3
def leaf_poly(x, k=3):
    p = [0] * k
    p[0] += 1
    p[x % k] += 1
    return p

polys = [leaf_poly(1), leaf_poly(2), leaf_poly(1)]
total = [1, 0, 0]
for p in polys:
    total = cyc_conv(total, p, 3)
print("số cách theo số dư của ba lá:", total, "-> số dư 0:", total[0], "(trừ tập rỗng ->", total[0] - 1, ")")

số cách theo số dư của ba lá: [3, 3, 2] -> số dư 0: 3 (trừ tập rỗng -> 2 )


## 5. Cách 1 — Brute Force: thử mọi tập con

**Ý tưởng.** Với `n ≤ 16` duyệt `2^n` tập con bằng bitmask: loại tập rỗng, kiểm tra từng nút được chọn có cha được chọn hay không, kiểm tra tổng chia hết `k`.

In [7]:
def brute(parent, nums, k):
    n = len(nums)
    count = 0
    for mask in range(1, 1 << n):
        if any(mask >> i & 1 and parent[i] >= 0 and mask >> parent[i] & 1 for i in range(n)):
            continue
        if sum(nums[i] for i in range(n) if mask >> i & 1) % k == 0:
            count += 1
    return count % MOD

for args, exp in EXAMPLES:
    got = brute(*args)
    print(args, "->", got)
    assert got == exp

([-1, 0, 1], [1, 2, 3], 3) -> 1
([-1, 0, 0, 0], [2, 1, 2, 1], 3) -> 2


**Độ phức tạp.** `O(2^n · n)`: dùng được tới `n ≈ 16`. **Nút thắt:** số tập con bùng nổ. Nhưng quyết định ở một nút chỉ phụ thuộc cha và con ⇒ gom theo (nút, đã chọn nút đó chưa, số dư).

## 6. Bước đột phá

Định nghĩa với mỗi nút `v` hai mảng độ dài `k` (**đếm cả tập rỗng**):

- `f0[v][s]` = số cách chọn tập con của **cây con `v`** thỏa không có cặp kề nhau, **không chọn `v`**, tổng `≡ s (mod k)`.
- `f1[v][s]` = như trên nhưng **có chọn `v`**.

**Quy tắc ghép (các con độc lập sau khi biết trạng thái của `v`):**

- Nếu **không chọn `v`**: mỗi con `c` tự do ⇒ dùng `g[c] = f0[c] + f1[c]` (đã đếm hết các khả năng của `c`).
  `f0[v] = g[c₁] ⊛ g[c₂] ⊛ ...` (⊛ là tích chập vòng).
- Nếu **chọn `v`**: mọi con **không được chọn** ⇒ dùng `f0[c]`. Sau khi ghép, dịch thêm `nums[v] % k` vì nút `v` đóng góp.
  `f1[v] = shift( f0[c₁] ⊛ f0[c₂] ⊛ ..., nums[v] % k )`.

Với lá: tích chập rỗng là `[1, 0, ..., 0]` (một cách, tổng 0) nên `f0 = [1,0..]`, `f1 = shift([1,0,..], x%k)`.

**Đáp án:** `f0[0][0] + f1[0][0] − 1` (trừ tập rỗng, vì `f0[0][0]` luôn đếm tập rỗng với tổng 0).

**Quy tắc:** *"nếu chọn cha thì con dùng `f0`; nếu không chọn cha thì con dùng `f0 + f1`"*.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP đệ quy với tích chập ngây thơ

**Giải thích từng đoạn.**

1. Dựng `children[v]` từ `parent`.
2. `dfs(v)` trả về cặp `(f0, f1)`:
   - `A0 = [1,0,..]`, `A1 = [1,0,..]` (tích rỗng);
   - với mỗi con `c`: `(a, b) = dfs(c)`; `A0 = conv(A0, (a + b) % MOD)`, `A1 = conv(A1, a)`;
   - `f0 = A0`, `f1 = shift(A1, nums[v] % k)`.
3. Kết quả `(f0[0] + f1[0] − 1) % MOD` tại gốc.

In [8]:
def dp_recursive(parent, nums, k):
    n = len(nums)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[parent[i]].append(i)

    def dfs(v):
        A0 = [0] * k; A0[0] = 1
        A1 = [0] * k; A1[0] = 1
        for c in children[v]:
            a, b = dfs(c)
            g = [(x + y) % MOD for x, y in zip(a, b)]
            A0 = cyc_conv(A0, g, k)
            A1 = cyc_conv(A1, a, k)
        r = nums[v] % k
        f1 = [0] * k
        for s in range(k):
            f1[(s + r) % k] = A1[s]
        return A0, f1

    f0, f1 = dfs(0)
    return (f0[0] + f1[0] - 1) % MOD

for args, exp in EXAMPLES:
    got = dp_recursive(*args)
    print(args, "->", got)
    assert got == exp

([-1, 0, 1], [1, 2, 3], 3) -> 1
([-1, 0, 0, 0], [2, 1, 2, 1], 3) -> 2


**Độ phức tạp:** mỗi cạnh làm hai tích chập `O(k²)` ⇒ `O(n·k²)`; với `n = 1000`, `k = 100`: `2 · 10^7` phép nhân-cộng. **Không gian:** `O(n·k)` (cộng độ sâu đệ quy `O(n)` — nguy hiểm khi `n = 1000` là đường thẳng vì Python giới hạn ~1000).

### 7.2. Cách 3 — DP lặp theo thứ tự `n−1 → 0`, tích chập bằng "dịch vòng"

**Giải thích từng đoạn.**

1. `acc0[v]`, `acc1[v]` khởi tạo `[1,0,..]` (tích rỗng cho cây con `v`).
2. Duyệt `v` từ `n−1` xuống `1`: lúc này mọi con của `v` đã ghép xong vào `acc0[v]`, `acc1[v]`:
   - `f0 = acc0[v]`, `f1 = dịch(acc1[v], nums[v] % k)`, `g = f0 + f1`;
   - ghép vào cha `p`: `acc0[p] = conv(acc0[p], g)`, `acc1[p] = conv(acc1[p], f0)`.
3. Với `v = 0`: tính đáp án như trên.

**Tăng tốc tích chập:** với mỗi `i` có `a[i] ≠ 0`, ta cần cộng `a[i]·b` đã **dịch vòng `i` ô** vào kết quả. Dịch vòng bằng cắt lát: `b[-i:] + b[:-i]`. Mỗi bước là một list comprehension trên `k` phần tử → nhanh hơn hai vòng `for` lồng nhau nhiều lần. Bỏ qua `a[i] = 0` (nhiều cây con thưa).

In [9]:
def cyc_conv_fast(a, b, k):
    res = [0] * k
    for i, ai in enumerate(a):
        if ai:
            rot = b[-i:] + b[:-i] if i else b       # rot[t] = b[(t - i) % k]
            res = [x + ai * y for x, y in zip(res, rot)]
    return [x % MOD for x in res]

# kiểm tra khớp tích chập ngây thơ
rng = random.Random(1)
for _ in range(200):
    k = rng.randint(1, 9)
    a = [rng.randint(0, MOD - 1) if rng.random() < 0.7 else 0 for _ in range(k)]
    b = [rng.randint(0, MOD - 1) for _ in range(k)]
    assert cyc_conv_fast(a, b, k) == cyc_conv(a, b, k)
print("cyc_conv_fast = cyc_conv")

def dp_iterative(parent, nums, k):
    n = len(nums)
    acc0 = [[1] + [0] * (k - 1) for _ in range(n)]
    acc1 = [[1] + [0] * (k - 1) for _ in range(n)]
    for v in range(n - 1, 0, -1):
        f0 = acc0[v]
        r = nums[v] % k
        f1 = acc1[v][-r:] + acc1[v][:-r] if r else acc1[v]       # dịch vòng r ô
        g = [(x + y) % MOD for x, y in zip(f0, f1)]
        p = parent[v]
        acc0[p] = cyc_conv_fast(acc0[p], g, k)
        acc1[p] = cyc_conv_fast(acc1[p], f0, k)
        acc0[v] = acc1[v] = None                                   # giải phóng
    r = nums[0] % k
    f1 = acc1[0][-r:] + acc1[0][:-r] if r else acc1[0]
    return (acc0[0][0] + f1[0] - 1) % MOD

for args, exp in EXAMPLES:
    got = dp_iterative(*args)
    print(args, "->", got)
    assert got == exp

cyc_conv_fast = cyc_conv
([-1, 0, 1], [1, 2, 3], 3) -> 1
([-1, 0, 0, 0], [2, 1, 2, 1], 3) -> 2


**Độ phức tạp:** `O(n·k²)` thời gian (hằng số nhỏ nhờ list comprehension); không gian `O(n·k)` (có thể giảm khi giải phóng mảng sau khi ghép). Không đệ quy.

Có thể nghĩ tới `FFT`/`NTT` để giảm `k²` xuống `k log k`, nhưng với `k ≤ 100` điều đó không cần thiết.

## 8. TRACE từng bước

Trace Cách 3 trên `parent = [-1, 0, 0, 1, 1]`, `nums = [3, 1, 2, 2, 4]`, `k = 3`.

Cây: `0` có con `1, 2`; `1` có con `3, 4`. Số dư của `nums` theo `k = 3`: nút `0:0`, `1:1`, `2:2`, `3:2`, `4:1`. Mô phỏng từ `v = 4` về `0` (`g = f0 + f1`):

| `v` | `f0` | `f1` (đã dịch) | `g` | sau khi gộp vào cha |
|---|---|---|---|---|
| 4 | `[1,0,0]` | `[0,1,0]` | `[1,1,0]` | `acc0[1] = [1,1,0]`, `acc1[1] = [1,0,0]` |
| 3 | `[1,0,0]` | `[0,0,1]` | `[1,0,1]` | `acc0[1] = [2,1,1]`, `acc1[1] = [1,0,0]` |
| 2 | `[1,0,0]` | `[0,0,1]` | `[1,0,1]` | `acc0[0] = [1,0,1]`, `acc1[0] = [1,0,0]` |
| 1 | `[2,1,1]` | `[0,1,0]` | `[2,2,1]` | `acc0[0] = [4,3,3]`, `acc1[0] = [2,1,1]` |
| 0 | `[4,3,3]` | `[2,1,1]` (dịch 0) | | đáp án `4 + 2 − 1 = 5` |

Hãy chạy hàm trace — nó in từng bước và đáp án cuối, rồi so khớp với brute force.

In [10]:
def trace_dp(parent, nums, k):
    n = len(nums)
    acc0 = [[1] + [0] * (k - 1) for _ in range(n)]
    acc1 = [[1] + [0] * (k - 1) for _ in range(n)]
    for v in range(n - 1, 0, -1):
        f0 = acc0[v]
        r = nums[v] % k
        f1 = acc1[v][-r:] + acc1[v][:-r] if r else acc1[v]
        g = [(x + y) % MOD for x, y in zip(f0, f1)]
        p = parent[v]
        print(f"nút {v} (giá trị {nums[v]}, dịch {r}): f0={f0} f1={f1} g=f0+f1={g}")
        acc0[p] = cyc_conv_fast(acc0[p], g, k)
        acc1[p] = cyc_conv_fast(acc1[p], f0, k)
        print(f"     gộp vào cha {p}: acc0[{p}]={acc0[p]}  acc1[{p}]={acc1[p]}")
    r = nums[0] % k
    f1 = acc1[0][-r:] + acc1[0][:-r] if r else acc1[0]
    print(f"gốc 0: f0={acc0[0]} f1={f1}")
    ans = (acc0[0][0] + f1[0] - 1) % MOD
    print("đáp án = f0[0] + f1[0] - 1 =", acc0[0][0], "+", f1[0], "- 1 =", ans)
    return ans

parent, nums, k = [-1, 0, 0, 1, 1], [3, 1, 2, 2, 4], 3
assert trace_dp(parent, nums, k) == brute(parent, nums, k)

nút 4 (giá trị 4, dịch 1): f0=[1, 0, 0] f1=[0, 1, 0] g=f0+f1=[1, 1, 0]
     gộp vào cha 1: acc0[1]=[1, 1, 0]  acc1[1]=[1, 0, 0]
nút 3 (giá trị 2, dịch 2): f0=[1, 0, 0] f1=[0, 0, 1] g=f0+f1=[1, 0, 1]
     gộp vào cha 1: acc0[1]=[2, 1, 1]  acc1[1]=[1, 0, 0]
nút 2 (giá trị 2, dịch 2): f0=[1, 0, 0] f1=[0, 0, 1] g=f0+f1=[1, 0, 1]
     gộp vào cha 0: acc0[0]=[1, 0, 1]  acc1[0]=[1, 0, 0]
nút 1 (giá trị 1, dịch 1): f0=[2, 1, 1] f1=[0, 1, 0] g=f0+f1=[2, 2, 1]
     gộp vào cha 0: acc0[0]=[4, 3, 3]  acc1[0]=[2, 1, 1]
gốc 0: f0=[4, 3, 3] f1=[2, 1, 1]
đáp án = f0[0] + f1[0] - 1 = 4 + 2 - 1 = 5


In [11]:
# Liệt kê các tập hợp lệ của cây trên để đối chiếu bằng mắt
n = len(nums)
for mask in range(1, 1 << n):
    S = [i for i in range(n) if mask >> i & 1]
    if any(parent[i] >= 0 and parent[i] in S for i in S):
        continue
    if sum(nums[i] for i in S) % k == 0:
        print(S, "tổng", sum(nums[i] for i in S))

[0] tổng 3
[1, 2] tổng 3
[2, 4] tổng 6
[3, 4] tổng 6
[0, 3, 4] tổng 9


## 9. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def countValidSubsets(self, parent: List[int], nums: List[int], k: int) -> int:
        MOD = 10**9 + 7
        n = len(nums)

        def conv(a, b):
            # tích chập vòng độ dài k: c[(i + j) % k] += a[i] * b[j]
            res = [0] * k
            for i, ai in enumerate(a):
                if ai:
                    rot = b[-i:] + b[:-i] if i else b
                    res = [x + ai * y for x, y in zip(res, rot)]
            return [x % MOD for x in res]

        acc0 = [[1] + [0] * (k - 1) for _ in range(n)]   # không chọn v: tích các (f0 + f1) của con
        acc1 = [[1] + [0] * (k - 1) for _ in range(n)]   # chọn v: tích các f0 của con (chưa cộng nums[v])

        for v in range(n - 1, 0, -1):                    # parent[v] < v nên con luôn được xử lý trước cha
            f0 = acc0[v]
            r = nums[v] % k
            f1 = acc1[v][-r:] + acc1[v][:-r] if r else acc1[v]
            g = [(x + y) % MOD for x, y in zip(f0, f1)]
            p = parent[v]
            acc0[p] = conv(acc0[p], g)
            acc1[p] = conv(acc1[p], f0)
            acc0[v] = acc1[v] = None

        r = nums[0] % k
        f1 = acc1[0][-r:] + acc1[0][:-r] if r else acc1[0]
        return (acc0[0][0] + f1[0] - 1) % MOD             # trừ tập rỗng

sol = Solution()
for args, exp in EXAMPLES:
    got = sol.countValidSubsets(*args)
    print(args, "->", got)
    assert got == exp

([-1, 0, 1], [1, 2, 3], 3) -> 1
([-1, 0, 0, 0], [2, 1, 2, 1], 3) -> 2


## 10. Vì sao thuật toán đúng

**Bất biến.** Sau khi xử lý xong mọi nút trong cây con của `v` (và chỉ chúng), `acc0[v]` bằng tích chập các `g[c] = f0[c] + f1[c]` qua các con `c`, và `acc1[v]` bằng tích chập các `f0[c]`. Từ đó `f0[v] = acc0[v]`, `f1[v] = dịch(acc1[v], nums[v] % k)`.

**Chứng minh.** Một tập độc lập trong cây con của `v` được xác định bởi (có chọn `v` hay không) và (tập độc lập trong từng cây con của các con).

- Nếu không chọn `v`: các cây con của con hoàn toàn độc lập và mỗi cây con có thể ở trạng thái bất kỳ (con được chọn hay không đều hợp lệ vì cặp (con, cha `v`) không có `v`) → số cách với tổng số dư `s` là tích chập các `g[c]`.
- Nếu chọn `v`: không con nào được chọn ⇒ mỗi con ở trạng thái `f0[c]`; tổng của tập = tổng các con + `nums[v]`, nên dịch `nums[v] % k`.

Tích chập vòng đúng vì tổng các số dư theo modulo `k` được bảo toàn: `(i + j) mod k`.

**Thứ tự lặp.** `parent[v] < v` nên các con của `v` (có chỉ số `> v`) đã gộp vào `acc*[v]` trước khi ta duyệt tới `v`.

**Trừ tập rỗng.** Tập rỗng không chọn `v = 0`, các con đều rỗng nên được đếm trong `f0[0][0]` với tổng `0`; nó không phải tập hợp lệ ("khác rỗng"). Mọi tập khác rỗng có tổng `≡ 0` đều được đếm đúng một lần.

## 11. Phân tích Time/Space

Đặt `n` là số nút, `k ≤ 100`.

| Bước | Thời gian |
|---|---|
| Mỗi cạnh `(v → cha)`: hai tích chập vòng | `O(k²)` |
| Dịch vòng `f1` cho mỗi nút | `O(k)` |
| Tổng | `O(n·k²)` |

Với `n = 1000`, `k = 100`: khoảng `2·10^7` phép nhân-cộng (list comprehension nên chạy trong ~1–2 giây trong Python).

**Không gian:** `O(n·k)` cho các mảng `acc0`, `acc1` (có thể giải phóng sau khi gộp vào cha, như trong code).

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, `k = 1`, cây đường thẳng, cây sao), phản ví dụ, và đối chiếu ngẫu nhiên giữa vét cạn, DP đệ quy ngây thơ, DP lặp, `Solution`.

In [13]:
def gen_parent(n, shape, rng):
    parent = [-1]
    for i in range(1, n):
        if shape == "path":
            parent.append(i - 1)
        elif shape == "star":
            parent.append(0)
        elif shape == "binary":
            parent.append((i - 1) // 2)
        else:
            parent.append(rng.randrange(i))
    return parent

sol = Solution()
for args, exp in EXAMPLES:
    assert sol.countValidSubsets(*args) == exp

assert sol.countValidSubsets([-1], [6], 3) == 1               # n = 1: tập {0} có tổng 6
assert sol.countValidSubsets([-1], [5], 3) == 0               # không có tập khác rỗng nào hợp lệ
assert sol.countValidSubsets([-1, 0], [1, 1], 1) == 2         # k = 1: mọi tập độc lập khác rỗng: {0}, {1}
assert sol.countValidSubsets([-1, 0, 0, 0], [1, 1, 1, 1], 1) == 8   # sao: 1 (gốc) + 7 (tập khác rỗng của 3 lá)
assert sol.countValidSubsets([-1, 0, 1, 2], [1, 1, 1, 1], 1) == 7   # đường 4 nút: 8 tập độc lập - rỗng
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [14]:
rng = random.Random(3939)
for _ in range(300):
    n = rng.randint(1, 13)
    k = rng.randint(1, 6)
    parent = gen_parent(n, rng.choice(["random", "path", "star", "binary"]), rng)
    nums = [rng.randint(1, 10**9) if rng.random() < 0.5 else rng.randint(1, 8) for _ in range(n)]
    exp = brute(parent, nums, k)
    assert dp_recursive(parent, nums, k) == exp, (parent, nums, k)
    assert dp_iterative(parent, nums, k) == exp, (parent, nums, k)
    assert sol.countValidSubsets(parent, nums, k) == exp, (parent, nums, k)
print("300 test nhỏ: 4 cách khớp nhau")

# lớn hơn: hai DP đối chiếu nhau (đủ lớn để modulo có tác dụng)
for _ in range(40):
    n = rng.randint(30, 120)
    k = rng.randint(1, 20)
    parent = gen_parent(n, rng.choice(["random", "path", "star", "binary"]), rng)
    nums = [rng.randint(1, 10**9) for _ in range(n)]
    assert dp_iterative(parent, nums, k) == sol.countValidSubsets(parent, nums, k)
print("40 test vừa khớp (dp_iterative = Solution)")

300 test nhỏ: 4 cách khớp nhau
40 test vừa khớp (dp_iterative = Solution)


In [15]:
# Thời gian ở giới hạn: n = 1000, k = 100
rng = random.Random(1)
nums = [rng.randint(1, 10**9) for _ in range(1000)]
for shape in ("random", "path", "star", "binary"):
    parent = gen_parent(1000, shape, rng)
    t0 = time.time()
    r = sol.countValidSubsets(parent, nums, 100)
    print(f"{shape:7s} n=1000 k=100 -> {r}  ({time.time() - t0:.2f}s)")

random  n=1000 k=100 -> 384261969  (0.13s)


path    n=1000 k=100 -> 869331782  (0.03s)


star    n=1000 k=100 -> 669378113  (0.50s)
binary  n=1000 k=100 -> 561586460  (0.12s)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên trừ tập rỗng

**Code sai:** `return (f0[0] + f1[0]) % MOD`.
**Vì sao sai:** tập rỗng có tổng `0` nên được đếm; đề yêu cầu **khác rỗng**.
**Code đúng:** `(f0[0] + f1[0] − 1) % MOD`.

In [16]:
def wrong_no_subtract(parent, nums, k):
    return (dp_iterative(parent, nums, k) + 1) % MOD       # tương đương quên trừ 1

print("sai :", wrong_no_subtract([-1, 0, 1], [1, 2, 3], 3), " đúng:", Solution().countValidSubsets([-1, 0, 1], [1, 2, 3], 3))

sai : 2  đúng: 1


### Lỗi 2 — Khi chọn cha vẫn cho con dùng `f0 + f1`

**Code sai:** `acc1[p] = conv(acc1[p], g)`.
**Vì sao sai:** nếu chọn `p` thì con **không** được chọn; dùng `f1` của con tạo cặp cha–con cùng được chọn.
**Code đúng:** `acc1[p] = conv(acc1[p], f0)`.

In [17]:
def wrong_parent_child(parent, nums, k):
    n = len(nums)
    acc0 = [[1] + [0] * (k - 1) for _ in range(n)]
    acc1 = [[1] + [0] * (k - 1) for _ in range(n)]
    for v in range(n - 1, 0, -1):
        f0 = acc0[v]; r = nums[v] % k
        f1 = acc1[v][-r:] + acc1[v][:-r] if r else acc1[v]
        g = [(x + y) % MOD for x, y in zip(f0, f1)]
        p = parent[v]
        acc0[p] = cyc_conv_fast(acc0[p], g, k)
        acc1[p] = cyc_conv_fast(acc1[p], g, k)          # SAI: phải là f0
    r = nums[0] % k
    f1 = acc1[0][-r:] + acc1[0][:-r] if r else acc1[0]
    return (acc0[0][0] + f1[0] - 1) % MOD

print("sai :", wrong_parent_child([-1, 0, 1], [1, 2, 3], 3), " đúng:", Solution().countValidSubsets([-1, 0, 1], [1, 2, 3], 3))

sai : 3  đúng: 1


### Lỗi 3 — Tích chập tuyến tính (không lấy modulo chỉ số)

**Code sai:** `c[i + j] += a[i]*b[j]` với `c` dài `2k − 1`, rồi chỉ đọc `c[0]`.
**Vì sao sai:** tổng `i + j = k, 2k` cũng ứng với số dư `0` nhưng bị bỏ.
**Code đúng:** chỉ số `(i + j) % k`.

In [18]:
def wrong_linear_conv(a, b, k):
    c = [0] * (2 * k - 1)
    for i in range(k):
        for j in range(k):
            c[i + j] += a[i] * b[j]
    return c[:k]          # SAI: mất các tổng i + j >= k

a, b = [1, 1, 0], [1, 0, 1]
print("sai :", wrong_linear_conv(a, b, 3), " đúng:", cyc_conv(a, b, 3))

sai : [1, 1, 1]  đúng: [2, 1, 1]


### Lỗi 4 — Quên `% k` khi dịch vòng hoặc quên `% MOD`

- Dùng `nums[v]` trực tiếp làm độ dịch → chỉ số vượt mảng (nums tới `10^9`). Phải `% k`.
- Không lấy `% MOD`: số cách có thể lớn hơn `10^9 + 7` rất nhiều (đường thẳng 1000 nút cho số cỡ `Fibonacci(1000)`), khác đáp án mong đợi.

In [19]:
# Ví dụ: số tập độc lập (k = 1) của đường thẳng n nút là Fibonacci: không lấy MOD thì khác
def count_no_mod(n):
    a, b = 1, 2           # a = số tập độc lập (kể cả rỗng) của path 0 nút... dùng đệ quy Fibonacci
    for _ in range(n - 1):
        a, b = b, a + b
    return b - 1          # trừ tập rỗng

n = 100
path = [-1] + list(range(n - 1))
print("đúng (modulo):", Solution().countValidSubsets(path, [1] * n, 1))
print("không modulo :", count_no_mod(n), "->", count_no_mod(n) % MOD, "(chỉ đúng sau khi lấy modulo)")
assert count_no_mod(n) % MOD == Solution().countValidSubsets(path, [1] * n, 1)

đúng (modulo): 470199268
không modulo : 927372692193078999175 -> 470199268 (chỉ đúng sau khi lấy modulo)


### Lỗi 5 — Đệ quy DFS trên cây đường thẳng `n = 1000`

Giới hạn đệ quy mặc định ~1000 khung; DFS đệ quy trên đường thẳng 1000 nút có thể gây `RecursionError`. Cách 3 (duyệt `n−1 → 0`) tránh hoàn toàn.

In [20]:
path = [-1] + list(range(999))
lim = sys.getrecursionlimit()
sys.setrecursionlimit(500)                       # mô phỏng ngăn xếp nhỏ
try:
    dp_recursive(path, [1] * 1000, 3)
    print("không lỗi (ngưỡng đủ lớn)")
except RecursionError:
    print("RecursionError với DFS đệ quy trên đường thẳng 1000 nút")
sys.setrecursionlimit(lim)
print("DP lặp:", dp_iterative(path, [1] * 1000, 3) == Solution().countValidSubsets(path, [1] * 1000, 3))

RecursionError với DFS đệ quy trên đường thẳng 1000 nút
DP lặp: True


## 14. Mẹo & nhận dạng pattern

- **"Đếm tập con thỏa ràng buộc theo cạnh trên cây"** → DP với trạng thái **(nút, đã chọn nút đó hay chưa)**; nếu thêm điều kiện về tổng → thêm **chiều số dư**.
- **Ghép các con độc lập** → nhân đa thức / **tích chập**; với tổng theo modulo thì là **tích chập vòng**.
- **Quy tắc cha–con:** chọn cha ⇒ con dùng "không chọn"; không chọn cha ⇒ con dùng "tất cả".
- Chỉ số cha < con (`parent[i] < i`) cho phép duyệt ngược **không đệ quy**.
- Luôn nhớ **loại tập rỗng** khi đề nói "khác rỗng" và điều kiện luôn đúng với tổng 0.
- Dịch vòng nhanh bằng cắt lát: `a[-r:] + a[:-r]` (khi `r > 0`).

## 15. Biến thể

1. **Cho phép chọn nút kề nhau** (không còn ràng buộc): đáp án là số tập con có tổng chia hết `k` — knapsack phẳng `O(n·k)` (không cần cây).
2. **Khoảng cách ≥ 3 thay vì ≥ 2:** trạng thái thêm "khoảng cách tới nút chọn gần nhất", như bài Subtree Inversion Sum II.
3. **Đếm theo tổng chính xác ≤ S nhỏ:** thay modulo bằng chiều tổng thật.
4. **Mỗi nút trọng số, tìm tổng lớn nhất không kề nhau** (House Robber III): cùng khung `f0/f1`, thay tích chập bằng cộng/`max`.
5. **Đếm cả tập rỗng:** bỏ phép trừ 1.

In [21]:
# Biến thể 4: House Robber III trên cùng khung f0/f1 (tổng lớn nhất không kề nhau)
def max_independent_sum(parent, nums):
    n = len(nums)
    f0 = [0] * n; f1 = [nums[i] for i in range(n)]
    for v in range(n - 1, 0, -1):
        p = parent[v]
        f0[p] += max(f0[v], f1[v])
        f1[p] += f0[v]
    return max(f0[0], f1[0])

print(max_independent_sum([-1, 0, 1], [1, 2, 3]), "(chọn 0 và 2: 1 + 3 = 4)")
print(max_independent_sum([-1, 0, 0, 0], [2, 1, 2, 1]), "(chọn các lá: 1 + 2 + 1 = 4 so với gốc 2)")

4 (chọn 0 và 2: 1 + 3 = 4)
4 (chọn các lá: 1 + 2 + 1 = 4 so với gốc 2)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập con | thử `2^n` tập, kiểm tra ràng buộc | `O(2^n · n)` | `O(1)` | kiểm chứng, `n ≤ 16` |
| 2. DP đệ quy, tích chập ngây thơ | `f0/f1` theo số dư, ghép con bằng hai vòng `for` | `O(n·k²)` | `O(n·k)` + ngăn xếp | dễ hiểu; cẩn thận đệ quy sâu |
| 3. DP lặp `n−1 → 0`, dịch vòng | cùng DP nhưng không đệ quy, tích chập bằng cắt lát | `O(n·k²)` (hằng số nhỏ) | `O(n·k)` | **lời giải nộp** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `cyc_conv([1, 1, 0], [1, 1, 0], 3)` bằng gì? (Ghép hai "không chọn hoặc chọn tổng 1".)

**Bài 2 (tính tay).** Cây đường thẳng `0 – 1 – 2` với `nums = [1, 1, 1]`, `k = 2`. Có bao nhiêu tập hợp lệ? (Liệt kê tập độc lập, xét tổng chẵn.)

**Bài 3 (điền TODO).** Hoàn thiện `my_shift(a, r)` — dịch vòng mảng `a` sang phải `r` ô (`r` có thể `0` hoặc `≥ len(a)`).

**Bài 4 (tự viết lại).** Viết `my_count_valid_subsets(parent, nums, k)` theo Cách 3, không nhìn code nộp.

**Bài 5 (biến thể).** Viết `count_with_size(parent, nums, k, m)`: đếm tập hợp lệ (độc lập, tổng chia hết `k`) có **đúng `m` nút** (vét cạn là đủ).

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_shift(a, r):
    # TODO: r %= len(a); trả về mảng sao cho phần tử a[s] chuyển tới vị trí (s + r) % len(a)
    pass

# Bài 4 — TODO
def my_count_valid_subsets(parent, nums, k):
    # TODO: acc0/acc1 khởi tạo [1,0,..]; duyệt v = n-1..1; gộp vào cha: acc0[p] ⊛= (f0+f1), acc1[p] ⊛= f0
    pass

# Bài 5 — TODO
def count_with_size(parent, nums, k, m):
    # TODO: bitmask hoặc itertools.combinations(range(n), m), kiểm tra độc lập + tổng chia hết k
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: hai đa thức `1 + x`; nhân lại.
- Mức 2: `(1 + x)² = 1 + 2x + x²`.
- Mức 3: chỉ số `0, 1, 2` → `[1, 2, 1]`.

**Bài 2.**
- Mức 1: tập độc lập khác rỗng của đường 3 nút: `{0}, {1}, {2}, {0,2}`.
- Mức 2: tổng `1, 1, 1, 2`.
- Mức 3: chỉ `{0,2}` có tổng chẵn → `1`.

**Bài 3.**
- Mức 1: tạo mảng mới độ dài `len(a)`.
- Mức 2: `res[(s + r) % n] = a[s]`.
- Mức 3: hoặc `a[-r:] + a[:-r]` khi `r != 0`.

**Bài 4.**
- Mức 1: xem lại `dp_iterative`.
- Mức 2: nhớ dịch `acc1` theo `nums[v] % k` trước khi gộp.
- Mức 3: đáp án `(acc0[0][0] + f1[0] − 1) % MOD`.

**Bài 5.**
- Mức 1: duyệt `combinations(range(n), m)`.
- Mức 2: kiểm tra `parent[i] not in S`.
- Mức 3: cộng tổng và `% k == 0`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert cyc_conv([1, 1, 0], [1, 1, 0], 3) == [1, 2, 1]
# Bài 2
assert Solution().countValidSubsets([-1, 0, 1], [1, 1, 1], 2) == 1 == brute([-1, 0, 1], [1, 1, 1], 2)

# Bài 3
def my_shift_ans(a, r):
    n = len(a)
    r %= n
    res = [0] * n
    for s in range(n):
        res[(s + r) % n] = a[s]
    return res

assert my_shift_ans([1, 2, 3, 4], 1) == [4, 1, 2, 3]
assert my_shift_ans([1, 2, 3, 4], 0) == [1, 2, 3, 4]
assert my_shift_ans([1, 2, 3, 4], 6) == [3, 4, 1, 2]
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_count_valid_subsets_ans(parent, nums, k):
    n = len(nums)
    acc0 = [[1] + [0] * (k - 1) for _ in range(n)]
    acc1 = [[1] + [0] * (k - 1) for _ in range(n)]
    for v in range(n - 1, 0, -1):
        f0 = acc0[v]
        f1 = my_shift_ans(acc1[v], nums[v])
        g = [(x + y) % MOD for x, y in zip(f0, f1)]
        p = parent[v]
        acc0[p] = cyc_conv(acc0[p], g, k)
        acc1[p] = cyc_conv(acc1[p], f0, k)
    f1 = my_shift_ans(acc1[0], nums[0])
    return (acc0[0][0] + f1[0] - 1) % MOD

rng = random.Random(31)
for _ in range(100):
    n = rng.randint(1, 12); k = rng.randint(1, 6)
    parent = gen_parent(n, "random", rng)
    nums = [rng.randint(1, 20) for _ in range(n)]
    assert my_count_valid_subsets_ans(parent, nums, k) == brute(parent, nums, k)

# Bài 5
def count_with_size_ans(parent, nums, k, m):
    n = len(nums)
    cnt = 0
    for S in combinations(range(n), m):
        sset = set(S)
        if any(parent[i] in sset for i in S if parent[i] >= 0):
            continue
        if sum(nums[i] for i in S) % k == 0:
            cnt += 1
    return cnt

assert count_with_size_ans([-1, 0, 0, 0], [2, 1, 2, 1], 3, 2) == 2
assert count_with_size_ans([-1, 0, 0, 0], [2, 1, 2, 1], 3, 1) == 0
assert count_with_size_ans([-1, 0, 1], [1, 2, 3], 3, 1) == 1
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tree knapsack theo số dư, `parent[i] < i`):**

```python
acc0 = [[1] + [0]*(k-1) for _ in range(n)]     # không chọn v
acc1 = [[1] + [0]*(k-1) for _ in range(n)]     # chọn v (chưa cộng nums[v])
for v in range(n - 1, 0, -1):
    f0 = acc0[v]; f1 = dịch(acc1[v], nums[v] % k)
    g = f0 + f1
    acc0[parent[v]] = conv_vong(acc0[parent[v]], g)
    acc1[parent[v]] = conv_vong(acc1[parent[v]], f0)
answer = (acc0[0][0] + dịch(acc1[0], nums[0] % k)[0] - 1) % MOD
```

**Checklist trước khi submit:**

- [ ] Chọn cha ⇒ con dùng `f0`; không chọn cha ⇒ con dùng `f0 + f1`.
- [ ] Tích chập **vòng** (chỉ số `% k`), không tuyến tính.
- [ ] Dịch theo `nums[v] % k` (không phải `nums[v]`).
- [ ] Trừ tập rỗng (`− 1`) rồi `% MOD`.
- [ ] `k = 1`: mảng độ dài 1, dịch/tích chập vẫn đúng.
- [ ] Không đệ quy; duyệt `n−1 → 0` nhờ `parent[i] < i`.
- [ ] Đã đối chiếu vét cạn `n ≤ 13` trên path/star/binary/random.
- [ ] Chạy thử `n = 1000`, `k = 100`.